In [2]:
# ==============================================================================
# CELL 1: Setup, Imports, and Helper Functions (FINAL CACHE-ONLY VERSION)
# ==============================================================================

# Install necessary libraries
!pip install -q pandas numpy scikit-learn lightgbm opencv-python Pillow transformers datasets sentence-transformers timm

import pandas as pd
import numpy as np
import os
import re
import math
import random
import warnings
import zipfile # <-- NEW IMPORT for unzipping
from scipy.sparse import hstack
from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb
import torch
from torchvision import transforms
import timm
from transformers import AutoTokenizer, AutoModel
# from google.colab import drive
from PIL import Image
from io import BytesIO

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Setup Complete. Libraries installed and imported.")

# --- Custom SMAPE Metric for LightGBM ---
def smape(y_true, y_pred):
    y_true_orig = np.exp(y_true) - 1
    y_pred_orig = np.exp(y_pred) - 1
    numerator = np.abs(y_pred_orig - y_true_orig)
    denominator = (np.abs(y_true_orig) + np.abs(y_pred_orig)) / 2
    return np.mean(numerator / denominator) * 100

def lgbm_smape(preds, train_data):
    labels = train_data.get_label()
    return 'SMAPE', smape(labels, preds), False

# --- Target Encoding Function (Out-of-Fold) ---
def target_encode(train_df, column, target_col, cv):
    oof_target_encoding = np.zeros(train_df.shape[0])
    for fold, (train_idx, val_idx) in enumerate(cv.split(train_df, groups=train_df[column])):
        mapping = train_df.loc[train_idx].groupby(column)[target_col].mean()
        oof_target_encoding[val_idx] = train_df.loc[val_idx, column].map(mapping).fillna(mapping.mean())
    return oof_target_encoding, train_df.groupby(column)[target_col].mean()

# --- Winsorization/Capping Helper ---
def winsorize(series, lower_quantile=0.01, upper_quantile=0.99):
    q_low = series.quantile(lower_quantile)
    q_high = series.quantile(upper_quantile)
    return series.clip(lower=q_low, upper=q_high)

print("Helper Functions Defined.")

Setup Complete. Libraries installed and imported.
Helper Functions Defined.


In [3]:
# ==============================================================================
# CELL 2: Data Loading and Preparation (MODIFIED FOR NEW PATHS AND UNZIP)
# ==============================================================================

# print("Mounting Google Drive and loading data...")
# drive.mount('/content/drive')

# 🚨 REPLACE THIS WITH YOUR ACTUAL BASE FOLDER PATH 🚨
# This is the folder containing 'dataset/' and 'image_cache.zip'
BASE_PATH = '/home/user/Downloads/student_resource/'

DATASET_PATH = BASE_PATH + 'dataset/'
IMAGE_CACHE_ZIP_PATH = DATASET_PATH + 'images_backup.zip' # Assumed zip file name
IMAGE_FOLDER = BASE_PATH + 'images_new/'            # Target extraction folder

# --- 💡 UNZIP IMAGE CACHE 💡 ---
print(f"Checking/Unzipping image cache from: {IMAGE_CACHE_ZIP_PATH}")
if not os.path.exists(IMAGE_FOLDER) or not os.listdir(IMAGE_FOLDER):
    os.makedirs(IMAGE_FOLDER, exist_ok=True)
    try:
        with zipfile.ZipFile(IMAGE_CACHE_ZIP_PATH, 'r') as zip_ref:
            # Extract to the dedicated image folder
            zip_ref.extractall(IMAGE_FOLDER)
        print(f"Successfully unzipped images to: {IMAGE_FOLDER}")
    except FileNotFoundError:
        print(f"ERROR: Image cache zip file not found at {IMAGE_CACHE_ZIP_PATH}. Proceeding, but image features will likely be all zeros.")
    except Exception as e:
        print(f"ERROR during unzipping: {e}. Image features might be incomplete.")
else:
    print(f"Image cache directory already exists and is populated at: {IMAGE_FOLDER}. Skipping unzip.")


# --- Load CSV Data (Using DATASET_PATH) ---
try:
    train_df = pd.read_csv(DATASET_PATH + 'train.csv')
    test_df = pd.read_csv(DATASET_PATH + 'test.csv')
except FileNotFoundError as e:
    print(f"ERROR: File not found. Double check your DATASET_PATH and spelling.")
    raise e

# --- Standardize Column Names ---
train_df = train_df.rename(columns={'sample_i': 'sample_id'})
test_df = test_df.rename(columns={'sample_i': 'sample_id'})

# --- Essential Data Checks/Cleaning ---
train_df['price'] = pd.to_numeric(train_df['price'], errors='coerce')
train_df['price'] = np.maximum(train_df['price'].fillna(0), 0.01)

print("\nReal Data Loaded and Columns Standardized.")
print(f"Train Shape: {train_df.shape}, Test Shape: {test_df.shape}")

Checking/Unzipping image cache from: /home/user/Downloads/student_resource/dataset/images_backup.zip
Successfully unzipped images to: /home/user/Downloads/student_resource/images_new/

Real Data Loaded and Columns Standardized.
Train Shape: (75000, 4), Test Shape: (75000, 3)


In [ ]:
# ==============================================================================
# CELL 3: Execution Pipeline, Training, and Prediction (MODIFIED PATHS)
# ==============================================================================

# --- IMAGE PROCESSING SETUP (Step 2: Image Preparation) ---
# IMAGE_FOLDER is already defined in CELL 2 (BASE_PATH + 'image_cache/')
# os.makedirs(IMAGE_FOLDER, exist_ok=True) # Removed as it's done during unzip setup
print(f"Image cache folder created/verified at: {IMAGE_FOLDER}")

# Image Transformation pipeline for EfficientNet-B0 (224x224 resize)
VISION_TRANSFORMS = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Combined dataframe
df_all = pd.concat([train_df.drop('price', axis=1, errors='ignore'), test_df], ignore_index=True)
is_train = df_all['sample_id'].isin(train_df['sample_id'])

# --- Phase 1: Feature Engineering (UNCHANGED) ---
train_df['y_log'] = np.log(train_df['price'] + 1); train_df['y_log_capped'] = winsorize(train_df['y_log']); y_log_train = train_df['y_log_capped'].values

df_all['catalog_content'] = df_all['catalog_content'].astype(str)
brand_extracted = df_all['catalog_content'].str.extract(r'(?:Item Name:\s*)?([A-Za-z0-9\s]+?)(?:\s*,|\s+[\d])', flags=re.IGNORECASE).fillna('')
df_all['Brand'] = brand_extracted[0].str.strip().str.split().str[0].fillna('UNKNOWN')
ipq_extracted = df_all['catalog_content'].str.extract(r'(?:Pack of|Case of|ct|count)\s*(\d+)', flags=re.IGNORECASE); df_all['IPQ'] = pd.to_numeric(ipq_extracted[0], errors='coerce').fillna(1)
weight_volume_re = r'(\d+\.?\d*)\s*(Ounce|oz|Fl Oz|ml|g|kg|L|lb|count)'; extracted_wv = df_all['catalog_content'].str.extract(weight_volume_re, flags=re.IGNORECASE)
df_all['Weight_Value'] = pd.to_numeric(extracted_wv[0], errors='coerce').fillna(0)

print("Generating TF-IDF Features...")
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=1000)
tfidf_matrix = tfidf.fit_transform(df_all['catalog_content']).astype(np.float32)

gkf_te = GroupKFold(n_splits=5); train_df_w_brand = df_all[is_train].copy(); train_df_w_brand['y_log_capped'] = y_log_train
train_df_w_brand['Brand_TE'], brand_te_mapping = target_encode(train_df_w_brand, 'Brand', 'y_log_capped', gkf_te)
df_all.loc[is_train, 'Brand_TE'] = train_df_w_brand['Brand_TE']
df_all.loc[~is_train, 'Brand_TE'] = df_all.loc[~is_train, 'Brand'].map(brand_te_mapping).fillna(brand_te_mapping.mean())
TABULAR_FEATURES = ['IPQ', 'Weight_Value', 'Brand_TE']
print("Phase 1 - Feature Engineering complete.")

# --- Phase 2: Feature Extraction (CACHE-ONLY) ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vision_model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0).to(DEVICE).eval()
text_tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
text_model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2').to(DEVICE).eval()

# FINAL FUNCTION: REAL Vision Feature Extraction - CACHE ONLY
def extract_vision_features_real(image_link, image_folder, model, device):
    local_path = os.path.join(image_folder, f"{image_link.split('/')[-1].split('?')[0]}.jpg")

    try:
        # CRITICAL: If the file is not found, we do NOT download. We assume the cache is complete.
        if not os.path.exists(local_path):
             return np.zeros(512, dtype=np.float32)

        # Load, Transform, and Predict (GPU process)
        img = Image.open(local_path).convert('RGB')
        img_tensor = VISION_TRANSFORMS(img).unsqueeze(0).to(device)

        with torch.no_grad():
            features = model(img_tensor).squeeze().cpu().numpy()
            return features

    except Exception as e:
        # Fallback for corrupted file
        return np.zeros(512, dtype=np.float32)

def extract_text_features(text, tokenizer, model, device):
    def mean_pooling(model_output, attention_mask):
        token_embeddings = model_output[0]; input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1); sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return sum_embeddings / sum_mask
    encoded_input = tokenizer(text, padding=True, truncation=True, return_tensors='pt', max_length=128).to(device)
    with torch.no_grad(): model_output = model(**encoded_input)
    sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
    return sentence_embeddings.cpu().numpy().flatten().astype(np.float32)

print("\nStarting CACHE-ONLY Vision/Text Feature Extraction (Expected to be fast)...")
VISION_FEATURES = np.stack(df_all['image_link'].apply(
    lambda x: extract_vision_features_real(x, IMAGE_FOLDER, vision_model, DEVICE)
).values)
TEXT_FEATURES = np.stack(df_all['catalog_content'].apply(lambda x: extract_text_features(str(x), text_tokenizer, text_model, DEVICE)).values)

# Step 7: Feature Fusion & Caching
tabular_dense = df_all[TABULAR_FEATURES].values.astype(np.float32); dl_features_dense = np.concatenate([VISION_FEATURES, TEXT_FEATURES], axis=1)
X_final = hstack([dl_features_dense, tabular_dense, tfidf_matrix]).tocsr()
X_train_final = X_final[is_train.values]; X_test_final = X_final[~is_train.values]
groups_train = df_all.loc[is_train, 'Brand'].values
print(f"Phase 2 - Feature Fusion complete. Train matrix shape: {X_train_final.shape}")


# Step 8: Model Training (LightGBM Regressor) - FULL 5-FOLD CV
lgb_params = {'objective': 'mae', 'metric': 'None', 'n_estimators': 1000, 'learning_rate': 0.05, 'num_leaves': 31, 'max_depth': 6, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 1, 'lambda_l1': 0.1, 'lambda_l2': 0.1, 'n_jobs': -1, 'seed': SEED, 'verbose': -1}
gkf_cv = GroupKFold(n_splits=5); oof_predictions = np.zeros(X_train_final.shape[0]); models = []

print("\nStarting 5-Fold Training...")
for fold, (train_idx, val_idx) in enumerate(gkf_cv.split(X_train_final, y_log_train, groups=groups_train)):
    X_tr, X_val = X_train_final[train_idx], X_train_final[val_idx]; y_tr, y_val = y_log_train[train_idx], y_log_train[val_idx]
    lgb_train = lgb.Dataset(X_tr, y_tr); lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)
    model = lgb.train(lgb_params, lgb_train, valid_sets=[lgb_val], feval=lgbm_smape, callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
    oof_predictions[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
    models.append(model)
    print(f"Fold {fold+1} finished. Best iteration: {model.best_iteration}")

# Step 9: Final Model Output (Train on all data)
avg_iterations = int(np.mean([m.best_iteration for m in models]))
final_model = lgb.train(lgb_params, lgb.Dataset(X_train_final, y_log_train), num_boost_round=avg_iterations)
MODEL_SAVE_PATH = BASE_PATH + 'lightgbm_final_model.txt' # <-- Updated path
final_model.save_model(MODEL_SAVE_PATH)

# --- Phase 3: High-Speed Inference on Test Set ---
y_log_pred = final_model.predict(X_test_final); price_pred = np.exp(y_log_pred) - 1; price_final = np.maximum(price_pred, 0.01)

# Step 13: Output - SAVED TO DRIVE
submission_df = pd.DataFrame({'sample_id': test_df['sample_id'], 'price': price_final})
SUBMISSION_SAVE_PATH = BASE_PATH + 'final_submission2.csv' # <-- Updated path
submission_df.to_csv(SUBMISSION_SAVE_PATH, index=False)

print("\n--- EXECUTION COMPLETE ---")
print(f"Prediction file saved permanently to: {SUBMISSION_SAVE_PATH}")
print(f"First 700 predictions from test.csv:\n{submission_df[:700]}")

# Export OOF predictions to memory for the next cell
oof_results = {'y_log_train': y_log_train, 'oof_predictions': oof_predictions}

Image cache folder created/verified at: /home/user/Downloads/student_resource/images_new/
Generating TF-IDF Features...
Phase 1 - Feature Engineering complete.

Starting CACHE-ONLY Vision/Text Feature Extraction (Expected to be fast)...


In [5]:
# ==============================================================================
# CELL 4 (REVISED): ACCURACY and OTHER METRICS REPORTING (WITHOUT SMAPE)
# ==============================================================================

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

# 1. Prepare True Target and Predictions on Original Scale
# Get log-transformed values from memory
y_log_true = oof_results['y_log_train']
y_log_pred = oof_results['oof_predictions']

# Inverse transform to the original currency/price scale
y_true_orig = np.exp(y_log_true) - 1
y_pred_orig = np.exp(y_log_pred) - 1

# 2. Calculate Requested Metrics (MAE, RMSE, R^2)

# MAE (Mean Absolute Error)
oof_mae_score = mean_absolute_error(y_true_orig, y_pred_orig)

# RMSE (Root Mean Squared Error)
oof_rmse_score = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))

# R^2 (Coefficient of Determination)
oof_r2_score = r2_score(y_true_orig, y_pred_orig)

print("\n=============================================")
print("✅ FINAL ACCURACY AND OTHER METRICS (OUT-OF-FOLD) ✅")
print("---------------------------------------------")
print("--- Metrics (On Original Price Scale) ---")
print(f"1. MAE (Mean Absolute Error): {oof_mae_score:.4f}")
print(f"2. RMSE (Root Mean Squared Error): {oof_rmse_score:.4f}")
print(f"3. R² (Coefficient of Determination): {oof_r2_score:.4f} (Closer to 1 is better)")
print("=============================================")


✅ FINAL ACCURACY AND OTHER METRICS (OUT-OF-FOLD) ✅
---------------------------------------------
--- Metrics (On Original Price Scale) ---
1. MAE (Mean Absolute Error): 12.3133
2. RMSE (Root Mean Squared Error): 22.0695
3. R² (Coefficient of Determination): 0.2588 (Closer to 1 is better)
